# Projeto Final - 01. Raw para Bronze (Delta)
Le todos os CSVs de ordens de producao da pasta RAW de uma vez (wildcard),
converte pra Delta e grava particionado por `linha_producao` -- e nao por
`id_maquina`, porque as consultas mais frequentes da API
(`producao-resumo?linha=L1`, `saude-linha?linha=L1`) filtram primeiro por linha.
Particionar por maquina geraria mais pastas pequenas (8 maquinas) sem ganho real
de performance pra esse padrao de consulta.

In [1]:
# Tag "parameters" -- e assim que o Data Factory/Synapse Pipeline injeta
# valores via Base parameters na Notebook Activity. Vazio = processa
# todos os arquivos da pasta de uma vez.
data_referencia = ""

StatementMeta(industry, 2, 2, Finished, Available, Finished, False)

In [2]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType
from pyspark.sql.functions import current_timestamp, input_file_name, regexp_extract
caminho_raw = "abfss://inicial-datalake@dlcursoazure.dfs.core.windows.net/raw/industria4.0/ordens-producao-*.csv"
caminho_bronze = "abfss://inicial-datalake@dlcursoazure.dfs.core.windows.net/BRONZE/ordens_producao_delta/"
schema_ordens = StructType([
    StructField("id_ordem", IntegerType(), True),
    StructField("data_producao", StringType(), True),
    StructField("turno", StringType(), True),
    StructField("linha_producao", StringType(), True),
    StructField("id_maquina", StringType(), True),
    StructField("produto", StringType(), True),
    StructField("quantidade_produzida", IntegerType(), True),
    StructField("quantidade_refugada", IntegerType(), True),
    StructField("duracao_minutos", IntegerType(), True),
])
# Wildcard "*" read the 7 files at once, like just one dataframe.
# Explicit schema avoids two readings (one to infer, another to load)
df_raw = spark.read.csv(caminho_raw, header=True, schema=schema_ordens)
print("Total lines read (all files):", df_raw.count())
df_raw.show(5)

StatementMeta(industry, 2, 3, Finished, Available, Finished, False)

Total lines read (all files): 168
+--------+-------------+-----+--------------+----------+------------+--------------------+-------------------+---------------+
|id_ordem|data_producao|turno|linha_producao|id_maquina|     produto|quantidade_produzida|quantidade_refugada|duracao_minutos|
+--------+-------------+-----+--------------+----------+------------+--------------------+-------------------+---------------+
|     145|   2026-07-07|Manha|            L1|       M01|Componente A|                 536|                 24|            477|
|     146|   2026-07-07|Tarde|            L1|       M01|Componente A|                 686|                 20|            464|
|     147|   2026-07-07|Noite|            L1|       M01|Componente A|                 516|                 18|            447|
|     148|   2026-07-07|Manha|            L1|       M02|Componente A|                 601|                 18|            423|
|     149|   2026-07-07|Tarde|            L1|       M02|Componente A|        

## Metadados de ingestao
Como agora sao varios arquivos num unico DataFrame, `input_file_name()` recupera
de qual arquivo cada linha veio -- essencial pra rastreabilidade quando o dado vem de
carga multipla.

In [3]:
df_bronze = (
    df_raw
    .withColumn("data_ingestao", current_timestamp())
    .withColumn("arquivo_origem", regexp_extract(input_file_name(), r"([^/]+\.csv)$", 1))
)
df_bronze.select("id_ordem", "linha_producao", "arquivo_origem", "data_ingestao").show(5, truncate=False)

StatementMeta(industry, 2, 4, Finished, Available, Finished, False)

+--------+--------------+--------------+--------------------------+
|id_ordem|linha_producao|arquivo_origem|data_ingestao             |
+--------+--------------+--------------+--------------------------+
|145     |L1            |              |2026-08-28 16:01:45.310356|
|146     |L1            |              |2026-08-28 16:01:45.310356|
|147     |L1            |              |2026-08-28 16:01:45.310356|
|148     |L1            |              |2026-08-28 16:01:45.310356|
|149     |L1            |              |2026-08-28 16:01:45.310356|
+--------+--------------+--------------+--------------------------+
only showing top 5 rows



## Gravando em Delta, particionado por linha_producao

In [4]:
(
    df_bronze.write
    .format("delta")
    .mode("overwrite")
    .partitionBy("linha_producao")
    .save(caminho_bronze)
)
print("Gravado em Delta na Bronze, particionado por linha_producao:", caminho_bronze)
print("Total de linhas gravadas:", df_bronze.count())

StatementMeta(industry, 2, 5, Finished, Available, Finished, False)

Gravado em Delta na Bronze, particionado por linha_producao: abfss://inicial-datalake@dlcursoazure.dfs.core.windows.net/BRONZE/ordens_producao_delta/
Total de linhas gravadas: 168


In [5]:
mssparkutils.notebook.exit(f"OK - {df_bronze.count()} linhas gravadas na bronze, particionado por linha_producao")

StatementMeta(industry, 2, 6, Finished, Available, Finished, False)

ExitValue: OK - 168 linhas gravadas na bronze, particionado por linha_producao